## 这一章主要使用mcp的方式调用
- gpt 调用MCP
- 实现一个简单的MCP

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI


load_dotenv("/etc/.env")

# 从环境变量读取配置
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
#print("endpoint:", endpoint)
api_key = os.getenv("AZURE_OPENAI_API_KEY")
# print("api_key:", api_key)
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT")
print("deployment_name:", deployment_name)

client = AzureOpenAI(
    api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
    api_version = "2025-03-01-preview",
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT']
)


response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "user", "content": "你能帮我写一个Python脚本吗？"},
        {"role": "assistant", "content": "当然可以！请告诉我你想要实现什么功能。"},
        {"role": "user", "content": "我想要一个计算斐波那契数列的脚本。"}
    ],
    max_tokens=2048
)

print("response:", response.choices[0].message.content)


deployment_name: gpt-4o
response: 好的！以下是一个计算斐波那契数列的Python脚本。它可以根据用户输入的数值，生成对应数量的斐波那契数列：

```python
def fibonacci(n):
    """
    生成斐波那契数列的前 n 项
    """
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    
    fib_sequence = [0, 1]
    for _ in range(2, n):
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence


if __name__ == "__main__":
    try:
        num = int(input("请输入您想生成的斐波那契数列项数: "))
        if num <= 0:
            print("请输入一个正整数！")
        else:
            result = fibonacci(num)
            print(f"斐波那契数列的前 {num} 项是: {result}")
    except ValueError:
        print("请输入一个有效的整数！")
```

### 如何运行此脚本
1. 在您的计算机上创建一个文件，例如 `fibonacci.py`。
2. 将上述代码复制并粘贴到文件中。
3. 使用终端或命令行运行脚本：
   ```
   python fibonacci.py
   ```
4. 根据提示输入您想要生成斐波那契数列的项数。

通过这个脚本，用户可以系统地计算任意长度的斐波那契数列。如果还有其他需求或改进，请随时告诉我！


In [ ]:
import subprocess

def call_azure_openai(prompt):
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=2048
    )
    return response.choices[0].message.content.strip()

def call_mcp_tool(prompt):
    """
    通过 MCP 的 docker tool 调用
    """
    try:
        # 这里以 settings.json 里的 docker 命令为例
        result = subprocess.run(
            ["docker", "run", "-i", "--rm", "mcp_server_time"],
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=15
        )
        if result.returncode == 0:
            return result.stdout.decode("utf-8").strip()
        else:
            return f"错误：{result.stderr.decode('utf-8')}"
    except Exception as e:
        return f"调用 MCP tool 失败: {e}"

# 示例调用
result = call_azure_openai("你好，当前准确的时间是多少，离圣诞节还有几天？")
print("Azure OpenAI 结果：", result)

mcp_tool_result = call_mcp_tool("你好，当前准确的时间是多少，离圣诞节还有几天？")
print("MCP Tool 结果：", mcp_tool_result)

ResourceNotFoundError: (404) Resource not found
Code: 404
Message: Resource not found